# Canada Wildfire 2023 — Data Cleaning

This notebook cleans the National Fire Database (NFDB) point data and produces
a filtered, classified dataset of uncontrolled wildfires in Canada for the year 2023.

**Workflow:**
1. Load raw NFDB data
2. Prepare & clean (rename, drop, recode)
3. Filter (year, cause, spatial outliers)
4. Classify fire size
5. Export to processed folder

## 0  |  Imports & Constants

In [1]:
from pathlib import Path

import pandas as pd

# ── File paths ──────────────────────────────────────────────────────────────
# Using pathlib ensures the paths work on Windows, macOS, and Linux.
# The notebook lives in notebooks/, so we go up one level to reach the project root.
DATA_DIR = Path("../data")
RAW_PATH       = DATA_DIR / "raw"       / "NFDB_point.csv"
PROCESSED_PATH = DATA_DIR / "processed" / "fire_2023_clean.csv"

# ── Analysis parameters ─────────────────────────────────────────────────────
TARGET_YEAR = 2023

# Canada is entirely in the western hemisphere, so a positive longitude flags
# a data-entry error (coordinates flipped to the wrong hemisphere).
MAX_VALID_LONGITUDE = 0

# Size-class thresholds in hectares (based on Canadian wildfire classification).
# Minimum observed: 200 ha | Maximum observed: 885 388 ha
SMALL_HA    =     500
MEDIUM_HA   =   1_000
BIG_HA      =  10_000
VERY_BIG_HA = 100_000

# Weights allow the size classes to drive a heat-map colour scale later.
# Exponential spacing (1 → 256) creates strong visual contrast between classes.
CLASS_WEIGHTS = {
    "small":     1,
    "medium":    2,
    "big":       4,
    "very big":  16,
    "mega fire": 256,
}

CATEGORY_ORDER = list(CLASS_WEIGHTS.keys())  # preserves logical, not alphabetical, order

## 1  |  Load Raw Data

In [2]:
# Fail fast: verify the file exists before starting any heavy processing.
if not RAW_PATH.is_file():
    raise FileNotFoundError(f"Raw data not found: {RAW_PATH.absolute()}")

fire_df = pd.read_csv(RAW_PATH, sep=";")

assert len(fire_df) > 0, "Loaded file is empty — check the CSV separator."
print(f"Loaded {len(fire_df):,} records from raw data.")

Loaded 15,206 records from raw data.


## 2  |  Prepare & Clean

In [3]:
# ── 2a  Rename columns to snake_case ────────────────────────────────────────
# Only the columns we actually need are renamed; the rest are dropped next.
COLUMN_RENAMES = {
    "SRC_AGENCY": "province",
    "LONGITUDE":  "longitude",
    "LATITUDE":   "latitude",
    "SIZE_HA":    "size_ha",
    "FIRE_ID":    "fire_id",
    "FID":        "fid",
    "CAUSE":      "cause",
    "YEAR":       "year",
}

fire_df = fire_df.rename(columns=COLUMN_RENAMES)

# ── 2b  Drop columns not used downstream ────────────────────────────────────
# errors="ignore" is intentional: if a column was already absent (e.g. after
# re-running this cell), we do not want a crash.
COLUMNS_TO_DROP = [
    "the_geom", "NFDBFIREID", "NAT_PARK", "FIRENAME",
    "REP_DATE",  "OUT_DATE",   "FIRE_TYPE", "RESPONSE",
    "PROTZONE",  "MORE_INFO",  "MONTH",     "DAY",
]

fire_df = fire_df.drop(columns=COLUMNS_TO_DROP, errors="ignore").copy()

# ── 2c  Decode province abbreviations ───────────────────────────────────────
PROVINCE_NAMES = {
    "ON": "Ontario",
    "BC": "British Columbia",
    "QC": "Quebec",
    "AB": "Alberta",
    "MB": "Manitoba",
    "SK": "Saskatchewan",
    "NS": "Nova Scotia",
    "NB": "New Brunswick",
    "NL": "Newfoundland and Labrador",
    "PE": "Prince Edward Island",
    "YT": "Yukon",
    "NT": "Northwest Territories",
    "NU": "Nunavut",
}

fire_df["province"] = fire_df["province"].map(PROVINCE_NAMES).fillna(fire_df["province"])

# ── 2d  Decode cause abbreviations ──────────────────────────────────────────
CAUSE_NAMES = {
    "H":    "Human",
    "H-PB": "Prescribed Burn",
    "U":    "Unknown",
    "N":    "Natural",
}

fire_df["cause"] = fire_df["cause"].map(CAUSE_NAMES).fillna(fire_df["cause"])

print("Columns after cleaning:", fire_df.columns.tolist())

Columns after cleaning: ['fid', 'province', 'fire_id', 'latitude', 'longitude', 'year', 'size_ha', 'cause']


## 3  |  Filter Records

In [4]:
# Prescribed burns are controlled events; this analysis focuses on uncontrolled fires only.
fire_df = fire_df[fire_df["cause"] != "Prescribed Burn"]

# Isolate the target year.
fire_2023_df = fire_df[fire_df["year"] == TARGET_YEAR].copy()

assert len(fire_2023_df) > 0, f"No records found for year {TARGET_YEAR}."
print(f"Uncontrolled fires in {TARGET_YEAR}: {len(fire_2023_df):,}")

# Remove spatial outliers: positive longitudes are impossible for Canada and
# indicate a data-entry error (sign flipped).
fire_2023_df = fire_2023_df[fire_2023_df["longitude"] <= MAX_VALID_LONGITUDE]

print(f"Records after removing spatial outliers: {len(fire_2023_df):,}")

Uncontrolled fires in 2023: 965
Records after removing spatial outliers: 964


## 4  |  Classify Fire Size

In [5]:
def classify_size(size_ha: float) -> str:
    """
    Classifies a wildfire by its burned area.

    Thresholds follow the Canadian wildfire size classification system.
    The boundaries use the module-level constants SMALL_HA, MEDIUM_HA,
    BIG_HA, and VERY_BIG_HA so the logic stays in one place.

    Parameters
    ----------
    size_ha : float
        Burned area in hectares.

    Returns
    -------
    str
        One of: 'small', 'medium', 'big', 'very big', 'mega fire'.
    """
    if size_ha < SMALL_HA:
        return "small"
    if size_ha < MEDIUM_HA:
        return "medium"
    if size_ha < BIG_HA:
        return "big"
    if size_ha < VERY_BIG_HA:
        return "very big"
    return "mega fire"


# Apply classification and sort largest fires to the top for easy inspection.
fire_2023_df = fire_2023_df.sort_values(by="size_ha", ascending=False)

fire_2023_df["size_class"] = fire_2023_df["size_ha"].apply(classify_size)

# Use an ordered Categorical so visualisation libraries respect the logical
# sequence (small → mega fire) instead of sorting alphabetically.
fire_2023_df["size_class"] = pd.Categorical(
    fire_2023_df["size_class"],
    categories=CATEGORY_ORDER,
    ordered=True,
)

fire_2023_df["class_weight"] = fire_2023_df["size_class"].map(CLASS_WEIGHTS)

print(fire_2023_df["size_class"].value_counts().sort_index())

size_class
small        198
medium       136
big          370
very big     224
mega fire     36
Name: count, dtype: int64


## 5  |  Export

In [6]:
# Create the output folder if it does not yet exist (safe on re-runs).
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

fire_2023_df.to_csv(PROCESSED_PATH, index=False)

print(f"Saved {len(fire_2023_df):,} records → {PROCESSED_PATH}")

Saved 964 records → ..\data\processed\fire_2023_clean.csv


,fid,province,fire_id,latitude,longitude,year,size_ha,cause,size_class,class_weight
843,NFDB_point.410,Quebec,20231080218,53.050500,-75.325500,2023,885388.2142,Natural,mega fire,256
516,NFDB_point.827,Northwest Territories,SS022-23,60.290967,-111.850550,2023,641421.0000,Natural,mega fire,256
905,NFDB_point.107,British Columbia,2023-G80280,57.618700,-122.012300,2023,619072.5000,Natural,mega fire,256
916,NFDB_point.231,Saskatchewan,23LX-SMITH,55.984833,-106.586650,2023,543976.0000,Natural,mega fire,256
903,NFDB_point.860,Northwest Territories,FS001-23,60.051400,-120.975900,2023,541766.0000,Natural,mega fire,256
...,...,...,...,...,...,...,...,...,...,...
451,NFDB_point.53,British Columbia,2023-R51245,53.853800,-128.593500,2023,200.0000,Natural,small,1
578,NFDB_point.325,Manitoba,NO057,55.855300,-95.645000,2023,200.0000,Natural,small,1
600,NFDB_point.848,Northwest Territories,ZF014-23,63.241500,-114.374000,2023,200.0000,Natural,small,1
10,NFDB_point.15191,Alberta,HWF161,59.731000,-119.751217,2023,200.0000,Natural,small,1
